# チャーン予測モデルのデプロイ（発展）

本編で構築したチャーン予測モデルを、AWS SageMaker のリアルタイム推論エンドポイントとしてデプロイし、
「顧客データを送ると解約確率＋施策判定が返る」状態を実装する。

**流れ**：環境確認 → 土台取得 → モデル学習 → S3保存 → 推論スクリプト作成 → デプロイ → 予測 → 削除

**使用環境**：SageMaker Studio（Image: Sagemaker Distribution 3.8.8）

---


## 0. 環境確認

SageMaker SDK が正しく読み込めるか確認する（`Session` が使えれば正常）。

In [ ]:
import sagemaker
print("Session" in dir(sagemaker))
print(sagemaker.__file__)

## 1. デプロイの土台を取得

デプロイに必要な3つを取得する。
- `session`：SageMaker を操作する窓口
- `role`：SageMaker が S3 等を操作する権限（実行ロール）
- `bucket`：モデルを保存する S3 バケット

In [ ]:
import sagemaker
import boto3

session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = session.default_bucket()

print("実行ロール:", role)
print("S3バケット:", bucket)

## 2. モデルの学習

本編と同じ前処理（型変換・欠損補完・ダミー化・層化分割）を行い、XGBoost を学習する。

※デプロイの題材に XGBoost を使う理由：SageMaker の組み込みアルゴリズムでデプロイ手順が用意されており扱いやすいため。

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# データ読み込み・前処理（本編と同じ）
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

y = (df["Churn"] == "Yes").astype(int)
X = df.drop(columns=["customerID", "Churn"])
X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# XGBoostを学習
model = XGBClassifier(eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)
print("学習完了")
print("特徴量の数:", X_train.shape[1])

## 3. モデルをファイルに保存

学習済みモデル（メモリ上）を、SageMaker が読める形式でファイルに書き出す。

In [ ]:
# XGBoostモデルを保存（SageMakerのXGBoostが読める形式）
model.save_model("xgboost-model")
print("モデルを保存しました")

# 中身を確認
import os
print("ファイルサイズ:", os.path.getsize("xgboost-model"), "バイト")

## 4. アーティファクト化（model.tar.gz）

モデルファイルを `model.tar.gz` に圧縮する。SageMaker はこの形式（モデルアーティファクト）でモデルを受け取る。

In [ ]:
import tarfile

# model.tar.gz に固める
with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add("xgboost-model")

print("アーティファクト(model.tar.gz)を作成しました")
print("サイズ:", os.path.getsize("model.tar.gz"), "バイト")

## 5. S3 にアップロード

アーティファクトを S3 に格納する。エンドポイントはここからモデルを読み込む。

In [ ]:
# model.tar.gz を S3 にアップロード
model_artifact = session.upload_data(
    path="model.tar.gz",
    bucket=bucket,
    key_prefix="churn-xgboost"
)

print("S3にアップロード完了")
print("モデルの場所:", model_artifact)

## 6. 推論スクリプト（inference.py）の作成

エンドポイント内でデータをどう処理して予測を返すかを、4つの関数で定義する。
- `model_fn`：起動時にモデルを読み込む
- `input_fn`：送信データを受け取り、**学習時のカラム名を付けて**モデルが読める形に変換
- `predict_fn`：予測する
- `output_fn`：結果（解約確率）を JSON で返す

※ `input_fn` でカラム名（`FEATURE_NAMES`）を付与するのがポイント。これが無いと XGBoost が
「学習時のフィールドが無い」と 500 エラーになる（実際に発生 → CloudWatch ログで原因特定して修正）。

In [ ]:
inference_code = '''
import os
import json
import xgboost as xgb
import pandas as pd

# 学習時のカラム名（順番も重要）
FEATURE_NAMES = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges", "gender_Male", "Partner_Yes", "Dependents_Yes", "PhoneService_Yes", "MultipleLines_No phone service", "MultipleLines_Yes", "InternetService_Fiber optic", "InternetService_No", "OnlineSecurity_No internet service", "OnlineSecurity_Yes", "OnlineBackup_No internet service", "OnlineBackup_Yes", "DeviceProtection_No internet service", "DeviceProtection_Yes", "TechSupport_No internet service", "TechSupport_Yes", "StreamingTV_No internet service", "StreamingTV_Yes", "StreamingMovies_No internet service", "StreamingMovies_Yes", "Contract_One year", "Contract_Two year", "PaperlessBilling_Yes", "PaymentMethod_Credit card (automatic)", "PaymentMethod_Electronic check", "PaymentMethod_Mailed check"]

def model_fn(model_dir):
    model = xgb.Booster()
    model.load_model(os.path.join(model_dir, "xgboost-model"))
    return model

def input_fn(request_body, content_type):
    if content_type == "text/csv":
        values = [float(x) for x in request_body.strip().split(",")]
        # カラム名付きのDataFrameにしてからDMatrixに
        df = pd.DataFrame([values], columns=FEATURE_NAMES)
        return xgb.DMatrix(df)
    else:
        raise ValueError(f"対応していない形式: {content_type}")

def predict_fn(input_data, model):
    return model.predict(input_data)

def output_fn(prediction, accept):
    return json.dumps({"churn_probability": float(prediction[0])}), accept
'''

with open("inference.py", "w") as f:
    f.write(inference_code)

print("inference.py を修正しました")

## 7. カラム名の確認

`inference.py` の `FEATURE_NAMES` が、学習時の特徴量名・順序と一致しているか確認する。

In [ ]:
print(list(X_train.columns))

## 8. デプロイ（エンドポイント作成）

修正した `inference.py` でモデルを定義し、エンドポイントを起動する。

⚠️ **このセルの実行から課金が始まる**（エンドポイントは削除するまで課金され続ける）。

In [ ]:
from sagemaker.xgboost.model import XGBoostModel

# 修正したinference.pyでモデルを再定義
xgb_model = XGBoostModel(
    model_data=model_artifact,
    role=role,
    entry_point="inference.py",
    framework_version="1.7-1",
    sagemaker_session=session,
)

# 再デプロイ（数分かかります・課金開始）
predictor = xgb_model.deploy(
    initial_instance_count=1,
    instance_type="ml.t2.medium",
    endpoint_name="churn-endpoint-v2"   # 名前を変える（前のと区別）
)
print("再デプロイ完了！")

## 9. 予測①：解約確率＋施策判定

テストデータから1件送り、解約確率を取得。本編で決めた閾値 0.274 で「解約リスクあり／継続見込み」を判定する。

In [ ]:
from sagemaker.serializers import CSVSerializer
import json

predictor.serializer = CSVSerializer()

# 1件取り出して送る
sample = X_test.iloc[0].astype(float).values
csv_input = ",".join([str(x) for x in sample])

# 予測
result = predictor.predict(csv_input)

# 結果を解釈
proba = json.loads(result[0][0])["churn_probability"]   # 解約確率を取り出す
threshold = 0.274   # 本編で決めた閾値

print(f"解約確率: {proba:.1%}")
print(f"判定閾値: {threshold:.1%}")
print("---")
if proba >= threshold:
    print(f"判定: 【解約リスクあり】→ 施策対象（フォロー・特典などを検討）")
else:
    print(f"判定: 【継続見込み】→ 特別な施策は不要")

## 10. 予測②：顧客別リスク診断レポート（可視化つき）

任意の顧客について、解約確率・主要因・推奨アクションをレポート化し、
各特徴量が「全体平均からどれだけズレているか」を横棒グラフで可視化する（赤=リスク要因／青=継続要因）。

※事前に日本語表示用ライブラリを入れる：`!pip install japanize-matplotlib`

In [ ]:
!pip install japanize-matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib  # 文字化け対策
import json

# ============================================================
# 🔄 【変更箇所】 調べたい顧客のインデックス番号をここに入力するだけ！
# ============================================================
target_index = 1909  # 例: 437番 や 2280番 など、調べたい番号に変えてください
# ============================================================

# 指定されたインデックスのデータを自動抽出
user_data = X_test.loc[target_index]
customer_id = f"No. {target_index}"  # レポート用の表記

# --- 1. SageMakerエンドポイントでの予測処理 ---
# 自動抽出したユーザーデータから予測用データを作成
sample = user_data.astype(float).values
csv_input = ",".join([str(x) for x in sample])

# 予測を実行
result = predictor.predict(csv_input)

# 結果の解釈
proba = json.loads(result[0][0])["churn_probability"]  # 解約確率
threshold = 0.274  # 閾値

# --- 2. 特徴量の統計計算 ---
mean_data = X_test.mean()
std_data = X_test.std()

# 平均からの乖離度（標準化）を計算
user_scaled = (user_data - mean_data) / std_data

# 特徴量名の日本語対応辞書
column_translation = {
    "SeniorCitizen": "シニア（高齢者）",
    "gender_Male": "性別：男性",
    "gender_Female": "性別：女性",
    "Partner_Yes": "配偶者：あり",
    "Partner_No": "配偶者：なし",
    "Dependents_Yes": "扶養家族：あり",
    "Dependents_No": "扶養家族：なし",
    "tenure": "契約期間（月数）",
    "PaperlessBilling_Yes": "ペーパーレス請求：利用",
    "PaperlessBilling_No": "ペーパーレス請求：非利用",
    "Contract_Month-to-month": "契約形態：月々更新",
    "Contract_One year": "契約形態：1年契約",
    "Contract_Two year": "契約形態：2年契約",
    "MonthlyCharges": "月額利用料金",
    "TotalCharges": "累計利用料金",
    "PhoneService_Yes": "電話サービス：加入",
    "PhoneService_No": "電話サービス：未加入",
    "MultipleLines_Yes": "複数回線：利用",
    "MultipleLines_No": "複数回線：利用なし",
    "MultipleLines_No phone service": "複数回線：電話未加入",
    "InternetService_DSL": "回線：DSL",
    "InternetService_Fiber optic": "回線：光回線 (Fiber optic)",
    "InternetService_No": "回線：ネット未契約",
    "OnlineSecurity_Yes": "セキュリティ：加入",
    "OnlineSecurity_No": "セキュリティ：未加入",
    "OnlineSecurity_No internet service": "セキュリティ：ネット未契約",
    "OnlineBackup_Yes": "バックアップ：加入",
    "OnlineBackup_No": "バックアップ：未加入",
    "OnlineBackup_No internet service": "バックアップ：ネット未契約",
    "DeviceProtection_Yes": "機器保証：加入",
    "DeviceProtection_No": "機器保証：未加入",
    "DeviceProtection_No internet service": "機器保証：ネット未契約",
    "TechSupport_Yes": "サポート窓口：加入",
    "TechSupport_No": "サポート窓口：未加入",
    "TechSupport_No internet service": "サポート窓口：ネット未契約",
    "StreamingTV_Yes": "TV配信：利用",
    "StreamingTV_No": "TV配信：利用なし",
    "StreamingTV_No internet service": "TV配信：ネット未契約",
    "StreamingMovies_Yes": "映画配信：利用",
    "StreamingMovies_No": "映画配信：利用なし",
    "StreamingMovies_No internet service": "映画配信：ネット未契約",
    "PaymentMethod_Bank transfer (automatic)": "決済：銀行振込（自動）",
    "PaymentMethod_Credit card (automatic)": "決済：クレジットカード（自動）",
    "PaymentMethod_Electronic check": "決済：電子チェック",
    "PaymentMethod_Mailed check": "決済：郵送チェック"
}

# グラフ用に日本語へ変換
japanese_labels = [column_translation.get(col, col) for col in user_scaled.index]

# 最も解約リスクを高めている原因を特定
top_risk_factor = user_scaled.idxmax()
risk_factor_ja = column_translation.get(top_risk_factor, top_risk_factor)

# --- 3. ビジネスサマリーの出力 ---
print("=" * 60)
print(f"📋 顧客解約リスク診断レポート (顧客ID: {customer_id})")
print("=" * 60)
print(f"【AI予測結果】 解約確率: {proba:.1%} （判定閾値: {threshold:.1%}）")
if proba >= threshold:
    print(f"【状況ステータス】 🔴 【要対応】解約リスクが基準値を上回っています。")
else:
    print(f"【状況ステータス】 🟢 【継続見込み】現時点で特別な対応は不要です。")
print(f"【主原因の分析】 「{risk_factor_ja}」 が全体平均を大きく上回っていることが要因です。")
print("-" * 60)
print("【推奨される次のアクション】")
if proba >= threshold:
    if proba >= 0.5:
        print("[即時対応] カスタマーサクセスより個別のお電話にて、不満点のリサーチとプラン見直しを提案。")
    else:
        print("[メールアプローチ] 次回更新時に使える「継続特典割引クーポン」を自動送付し、離脱を防止。")
else:
    print("[定期フォロー] 良好な関係維持のため、定期的な新機能のご案内メルマガを配信。")
print("=" * 60)
print("診断詳細（全体平均との比較）")

# --- 4. 視覚化グラフの描画 ---
plt.figure(figsize=(10, 10))
colors = ['#E63946' if x > 0 else '#4A90E2' for x in user_scaled.values]

sns.barplot(x=user_scaled.values, y=japanese_labels, hue=japanese_labels, palette=colors, legend=False)

plt.title(f"顧客 {customer_id} の特徴分析（中央の点線が『一般的な顧客』）", fontsize=13, fontweight='bold', pad=15)
plt.xlabel("← 継続しやすい傾向 ｜ 解約リスクを高める傾向 →\n（全体平均からのズレの大きさ）", fontsize=11)
plt.axvline(x=0, color='#666666', linestyle='--', linewidth=1.5)
plt.grid(axis='x', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()

## 11. エンドポイントの削除（課金停止）

⚠️ **最重要**：エンドポイントは削除するまで課金され続けるため、動作確認が終わったら必ず削除する。
削除後、一覧が空になったことを確認してコスト管理を完了する。

In [ ]:
# エンドポイントを削除（課金停止）
predictor.delete_endpoint()
print("エンドポイントを削除しました")

# 削除できたか確認
import boto3
sm = boto3.client("sagemaker")
endpoints = sm.list_endpoints()["Endpoints"]
if endpoints:
    print("まだ残っているエンドポイント:", [e["EndpointName"] for e in endpoints])
else:
    print("エンドポイントは全て削除済み（課金停止）")